In [1]:
from kedro.framework.session import KedroSession

session = KedroSession.create()
context = session.load_context()
catalog = context.catalog

                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=918309;file:///opt/homebrew/anaconda3/envs/credit-risk-kedro/lib/python3.12/site-packages/kedro_telemetry/plugin.py\plugin.py]8;;\:]8;id=796497;file:///opt/homebrew/anaconda3/envs/credit-risk-kedro/lib/python3.12/site-packages/kedro_telemetry/plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

In [2]:
import pandas as pd

df = catalog.load("raw_loans")
print(df.head())

print(df.columns)

[04/08/26 10:50:45] INFO     Loading data from raw_loans (CSVDataset)...                       ]8;id=956017;file:///opt/homebrew/anaconda3/envs/credit-risk-kedro/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=359071;file:///opt/homebrew/anaconda3/envs/credit-risk-kedro/lib/python3.12/site-packages/kedro/io/data_catalog.py#1048\1048]8;;\

[04/08/26 10:50:55] WARNING  /opt/homebrew/anaconda3/envs/credit-risk-kedro/lib/python3.12/site-pac ]8;id=86190;file:///opt/homebrew/anaconda3/envs/credit-risk-kedro/lib/python3.12/warnings.py\warnings.py]8;;\:]8;id=713201;file:///opt/homebrew/anaconda3/envs/credit-risk-kedro/lib/python3.12/warnings.py#112\112]8;;\
                             kages/kedro_datasets/pandas/csv_dataset.py:171: DtypeWarning: Columns                 
                             (0,19,49,59,118,129,130,131,134,135,136,139,145,146,147) have mixed                   
                             types. Specify dtype option on import or set low_memory=False.                        
                               return pd.read_csv(load_path, **self._load_args)                                    
                                                                                                                   

         id  member_id  loan_amnt  funded_amnt  funded_amnt_inv        term  \
0  68407277        NaN     3600.0       3600.0           3600.0   36 months   
1  68355089        NaN    24700.0      24700.0          24700.0   36 months   
2  68341763        NaN    20000.0      20000.0          20000.0   60 months   
3  66310712        NaN    35000.0      35000.0          35000.0   60 months   
4  68476807        NaN    10400.0      10400.0          10400.0   60 months   

   int_rate  installment grade sub_grade  ... hardship_payoff_balance_amount  \
0     13.99       123.03     C        C4  ...                            NaN   
1     11.99       820.28     C        C1  ...                            NaN   
2     10.78       432.66     B        B4  ...                            NaN   
3     14.85       829.90     C        C5  ...                            NaN   
4     22.45       289.91     F        F1  ...                            NaN   

  hardship_last_payment_amount disbursement_

In [3]:
df.columns[:30]


Index(['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv',
       'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title',
       'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
       'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose',
       'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs',
       'earliest_cr_line', 'fico_range_low', 'fico_range_high',
       'inq_last_6mths'],
      dtype='object')

In [4]:
df["loan_status"].unique()


array(['Fully Paid', 'Current', 'Charged Off', 'In Grace Period',
       'Late (31-120 days)', 'Late (16-30 days)', 'Default', nan,
       'Does not meet the credit policy. Status:Fully Paid',
       'Does not meet the credit policy. Status:Charged Off'],
      dtype=object)

In [5]:
df.shape

(2260701, 151)

In [6]:
df = df[df["loan_status"].isin(["Fully Paid", "Charged Off"])]

In [7]:
df.shape

(1345310, 151)

In [8]:
df['target'] = df['loan_status'].map({'Fully Paid': 0,'Default':1})
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term,target
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,0.0
5,68426831,NaN,11950.0,11950.0,11950.0,36 months,13.44,405.18,C,C3,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [9]:
selected_features = [
        "loan_amnt",
        "term",
        "int_rate",
        "grade",
        "emp_length",
        "home_ownership",
        "annual_inc",
        "verification_status",
        "purpose",
        "dti",
        "fico_range_low",
        "fico_range_high",
        "delinq_2yrs",
        "inq_last_6mths",
        "target"
    ]

df = df[selected_features]
df.head()

,loan_amnt,term,int_rate,grade,emp_length,home_ownership,annual_inc,verification_status,purpose,dti,fico_range_low,fico_range_high,delinq_2yrs,inq_last_6mths,target
0,3600.0,36 months,13.99,C,10+ years,MORTGAGE,55000.0,Not Verified,debt_consolidation,5.91,675.0,679.0,0.0,1.0,0.0
1,24700.0,36 months,11.99,C,10+ years,MORTGAGE,65000.0,Not Verified,small_business,16.06,715.0,719.0,1.0,4.0,0.0
2,20000.0,60 months,10.78,B,10+ years,MORTGAGE,63000.0,Not Verified,home_improvement,10.78,695.0,699.0,0.0,0.0,0.0
4,10400.0,60 months,22.45,F,3 years,MORTGAGE,104433.0,Source Verified,major_purchase,25.37,695.0,699.0,1.0,3.0,0.0
5,11950.0,36 months,13.44,C,4 years,RENT,34000.0,Source Verified,debt_consolidation,10.20,690.0,694.0,0.0,0.0,0.0


In [10]:
df.shape

(1345310, 15)